# json库
## 文件的读取与打印
* 读取：```js.load()```
    *
* 打印：```js.dump()```
* ```with open("vectorData.json",'r',encoding='utf-8')```
* 这句中的r/w是读取/写入控制符
    * with关键字可以让python自动关闭文件
## 字符串的转换
* python->json ```js.dumps()```
* json->python ```js.loads()```

In [1]:
import json

import numpy as np
import pandas as pd

# 读取.json文件
with open("vectorData.json",'r',encoding='utf-8') as f:
    data = f.read()
    print(data)


[
    {
        "group_name": "2d_task_1",
        "vectors": [[1,3],[1,2],[2,4],[3,1],[4,3],[5,5],[6,2],[7,7],[8,6],[9,8],[10,9]],
        "ori_axis": [[1,0],[0,1]],
        "tasks": [
            { "type": "axis_angle" },
            { "type": "change_axis", "obj_axis": [[2,1],[1,2]] },
            { "type": "area" },
            { "type": "axis_projection" },
            { "type": "axis_angle"}
        ]
    },
    {
        "group_name": "2d_task_2",
        "vectors": [[1,1],[2,0],[3,5],[4,2],[5,7],[6,4],[7,9],[8,6],[9,1],[10,8],[11,3],[12,10]],
        "ori_axis": [[1,1],[1,-1]],
        "tasks": [
            { "type": "area" },
            { "type": "axis_projection" },
            { "type": "change_axis", "obj_axis": [[3,2],[2,-3]] },
            { "type": "axis_projection"},
            { "type": "change_axis", "obj_axis": [[1,0],[0,1]] },
            { "type": "area"},
            { "type": "axis_angle" }
        ]
    },
    {
        "group_name": "2d_task_3",
        "vec

### 接下来准备正式接收数据

In [2]:
    data = json.loads(data)

    # 摊平数据
    vector_data = pd.json_normalize(
        data,
        # 定义要展开的层级
        record_path=['tasks'],
        # 定义要保留的外层字段
        meta = ['group_name', 'vectors', 'ori_axis'],
        errors='ignore',
    )
    # print(vector_data.info())
    # print(vector_data['type'].head())

    # 由于task中的各种东西过于混乱，我仅仅保留了目标坐标轴向量
    vector_data_clean = vector_data.dropna()
    vector_data_clean = vector_data_clean.drop('type',axis=1).copy()
    # 这里删掉这一列是因为我在后面类的定义里可以直接提取维度数
    vector_data_clean = vector_data_clean.drop('group_name',axis=1).copy()
    print(vector_data_clean.info())
    print(vector_data_clean.head())

    # 下面定义进行四种运算的类
    class AxCaculator:
        def __init__(self, ori_axis, obj_axis, vectors):
            # 将传入的嵌套列表转化为 numpy 数组，方便进行矩阵和向量运算
            # 此时vector被转换为大矩阵，ori/obj向量被转换为了
            # 假设 axis 矩阵的每一行是一个基向量
            self.ori_axis = np.array(ori_axis, dtype=float)
            self.obj_axis = np.array(obj_axis, dtype=float)
            self.vectors = np.array(vectors, dtype=float)

            # 获取维度确定初始的坐标轴向量维度 n
            self.n_dim = self.ori_axis.shape[0]

        # 坐标系转移（接受dataFrame的ori_axis, obj_axis, vectors、在维度为n维的情况下仍成立）
            # 转换公式：数量积/轴向量的模的平方(非正交的基不能使用该方法)
            # 转换： 矩阵的逆
        # 将vector转换为标准坐标系下的vector
        def to_standard(self):
            # 用矩阵乘法
            standard_vector = self.ori_axis @ self.vectors
            # 返回标准坐标系下的vectors
            return standard_vector

        # 检查新的坐标轴向量是否合法（所有的向量需要线性无关）
        def check_validity(self):
            # 坐标轴向量组成的一定是矩阵
            if np.linalg.det(self.obj_axis) == 0:
                print("坐标轴向量线性相关，无法将其作为坐标轴向量进行运算！")
                return False
            return True

        # 再将其转换为新坐标系下的向量
        # 这里使用了上面两个模块
        def obj_vector(self):
            if self.check_validity():
                standard_vec = self.to_standard()
                return standard_vec @ self.obj_axis
            else:
                return None

        # 在新坐标系下完成以下内容
        # 坐标系投影
            # 这个是求矩阵的点积然后再除以坐标轴向量即可（这里算的是标准坐标系下的绝对的投影长度）
            # 要除以坐标轴向量的长度（在点积分之后计算，非线性计算，不好复原）

            #下面是步骤
            # 1.原坐标要转到标准坐标系下（调用前面的转换模块）
            # 2.点积（新建一个模块）（每个标准vector矩阵与一个坐标轴相乘）
                # 为了点积，我们需要将坐标轴向量矩阵转置，这样才是行*行
                # 设n*m @ (m*m)T -> n*m(每一行中的每一个数都是和一个坐标轴点积之后的值)
                # 为了直接算出值，对坐标轴向量进行归一化（即长度为1）（非线性）
        def shade_vector(self):
            if self.check_validity():
                # 这里进行归一化 np.linalg.norm(self.obj_axis, axis=1, keepdims=True)求模长
                standard_axis_obj = (self.obj_axis / np.linalg.norm(self.obj_axis, axis=1, keepdims=True))
                return self.to_standard() @ standard_axis_obj
            else:
                return None
            # 3.除以坐标轴长度求得弧度角的大小（弃用）

        # 坐标系夹角
            # 将vector&目标轴向量归一化后转置点积分
            # 用cos值反求角度值
        def theta_axis(self):
            if self.check_validity():
                standard_vec = self.to_standard()
                # 这里进行归一化 np.linalg.norm(self.obj_axis, axis=1, keepdims=True)求模长
                standard_axis_obj = np.linalg.norm(self.obj_axis, axis=1, keepdims=True)
                standard_axis_vec = np.linalg.norm(standard_vec, axis=1, keepdims=True)
                cos_theta = (standard_axis_obj @ standard_axis_vec)
                # 反求
                return np.arccos(cos_theta)
            else:
                return None


        # 坐标系围成的高维体积（格拉姆矩阵的行列式的开方）
            # 即求
        def axis_value(self):
            if self.check_validity():
                # 这里相当于将所有矩阵的向量值平方
                # 所有向量的两两内积
                gram_matrix = self.obj_axis @ self.obj_axis.T

                # 计算这些向量围成的广义体积（面积的高维推广）
                # 对格拉姆矩阵求行列式后为体积的平方
                volume = np.sqrt(np.linalg.det(gram_matrix))
                return volume
            else:
                return None




<class 'pandas.core.frame.DataFrame'>
Index: 21 entries, 1 to 85
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   obj_axis  21 non-null     object
 1   vectors   21 non-null     object
 2   ori_axis  21 non-null     object
dtypes: object(3)
memory usage: 672.0+ bytes
None
             obj_axis                                            vectors  \
1    [[2, 1], [1, 2]]  [[1, 3], [1, 2], [2, 4], [3, 1], [4, 3], [5, 5...   
7   [[3, 2], [2, -3]]  [[1, 1], [2, 0], [3, 5], [4, 2], [5, 7], [6, 4...   
9    [[1, 0], [0, 1]]  [[1, 1], [2, 0], [3, 5], [4, 2], [5, 7], [6, 4...   
13   [[1, 1], [1, 0]]  [[0, 5], [1, 4], [2, 3], [3, 2], [4, 1], [5, 0...   
16   [[1, 0], [0, 1]]  [[0, 5], [1, 4], [2, 3], [3, 2], [4, 1], [5, 0...   

             ori_axis  
1    [[1, 0], [0, 1]]  
7   [[1, 1], [1, -1]]  
9   [[1, 1], [1, -1]]  
13   [[2, 3], [3, 2]]  
16   [[2, 3], [3, 2]]  
